In [3]:
import pandas as pd
import glob, os

DEMOGRAPHIC_PATH = "../data/demographic"

# Diagnóstico 1: ¿glob encuentra los archivos?
files = glob.glob(os.path.join(DEMOGRAPHIC_PATH, "*.xlsx"))
print(f"Archivos encontrados: {len(files)}")
for f in files:
    print(f"  - {f}")

Archivos encontrados: 2
  - ../data/demographic/DCD-area-proypoblacion-Mun-2005-2017_VP.xlsx
  - ../data/demographic/PPED-AreaMun-2018-2042_VP.xlsx


In [5]:
!pip install --upgrade openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
  Attempting uninstall: openpyxl
    Found existing installation: openpyxl 3.0.10
    Uninstalling openpyxl-3.0.10:
      Successfully uninstalled openpyxl-3.0.10


In [1]:
import pandas as pd
import glob, os

DEMOGRAPHIC_PATH = "../data/demographic"

files = glob.glob(os.path.join(DEMOGRAPHIC_PATH, "*.xlsx"))
df_list = []

for file in files:
    print(f"Cargando: {file}")
    df = pd.read_excel(file)
    df.columns = df.columns.str.lower().str.strip()
    
    # Filtrar solo total municipal (no cabecera/rural)
    df = df[df["area geografica"].str.upper() == "TOTAL"]
    
    df = df.rename(columns={
        "mpio": "cod_mpio",
        "año": "year",
        "población": "poblacion"
    })
    
    # Limpiar formato numérico (vienen con puntos como separador de miles)
    df["poblacion"] = (
        df["poblacion"]
        .astype(str)
        .str.replace(".", "", regex=False)
        .astype(float)
    )
    
    df["cod_mpio"] = df["cod_mpio"].astype(str).str.zfill(5)
    df["year"] = df["year"].astype(int)
    
    df_list.append(df[["cod_mpio", "year", "poblacion"]])

df_dane = pd.concat(df_list, ignore_index=True)
df_dane = df_dane.drop_duplicates(["cod_mpio", "year"])

# Filtrar solo los años que necesitas (2016-2024)
df_dane = df_dane[df_dane["year"].isin([2016, 2017, 2018, 2019, 2023, 2024])]

print(f"Filas finales: {len(df_dane)}")
df_dane.to_parquet("poblacion.parquet", compression="gzip")

Cargando: ../data/demographic/DCD-area-proypoblacion-Mun-2005-2017_VP.xlsx
Cargando: ../data/demographic/PPED-AreaMun-2018-2042_VP.xlsx
Filas finales: 6736


In [2]:
import pandas as pd

df_check = pd.read_parquet("poblacion.parquet")
print("Shape:", df_check.shape)
print("\nDtypes:")
print(df_check.dtypes)
print("\nMuestra:")
print(df_check.head())
print("\nAños presentes:")
print(sorted(df_check["year"].unique()))
print("\nMunicipios únicos:", df_check["cod_mpio"].nunique())
print("\nNulos:")
print(df_check.isna().sum())
print("\nRango población:")
print(df_check["poblacion"].describe())

Shape: (6736, 3)

Dtypes:
cod_mpio      object
year           int64
poblacion    float64
dtype: object

Muestra:
      cod_mpio  year  poblacion
12342    05001  2016  2351077.0
12343    05002  2016    20534.0
12344    05004  2016     2629.0
12345    05021  2016     4620.0
12346    05030  2016    29394.0

Años presentes:
[2016, 2017, 2018, 2019, 2023, 2024]

Municipios únicos: 1123

Nulos:
cod_mpio     0
year         0
poblacion    0
dtype: int64

Rango población:
count    6.736000e+03
mean     4.401799e+04
std      2.585396e+05
min      0.000000e+00
25%      6.629250e+03
50%      1.290700e+04
75%      2.747425e+04
max      7.918660e+06
Name: poblacion, dtype: float64


In [3]:
ceros = df_check[df_check["poblacion"] == 0]
print(f"Filas con población 0: {len(ceros)}")
print(ceros.head(20))
print(f"\nMunicipios únicos con 0: {ceros['cod_mpio'].nunique()}")
print(f"\nPoblación baja (<100):")
print(df_check[df_check["poblacion"] < 100].sort_values("poblacion").head(20))

Filas con población 0: 5
      cod_mpio  year  poblacion
15181    27493  2018        0.0
16304    27493  2019        0.0
20796    27493  2023        0.0
21303    94663  2023        0.0
22426    94663  2024        0.0

Municipios únicos con 0: 2

Población baja (<100):
      cod_mpio  year  poblacion
15181    27493  2018        0.0
16304    27493  2019        0.0
20796    27493  2023        0.0
21303    94663  2023        0.0
22426    94663  2024        0.0
